# Swarm ES in JAX: throughput scaling and seed sensitivity of evolution

strategies over thousands of vmapped environments

Aryan Singh Chandel (Rustamji Institute of Technology)  
November 25, 2026

We implement OpenAI-style evolution strategies (ES) entirely as jitted JAX functions. Environments are pure functions, an episode is one `lax.scan`, and a generation is `vmap` over policies and episodes. We study two questions on two small control tasks. First, how does environment throughput scale with the number of parallel environments on a CPU, a T4 GPU and a TPU? Second, how reliable is ES on the classic pendulum swing-up? On a T4 the drone task reaches about 15 million environment steps per second at 65,536 parallel environments, about 34 times the CPU, and a full 500-generation drone training run takes 1.5 minutes. On the pendulum, a hand-written energy controller confirms the task is solvable in our implementation, yet ES is strongly bimodal across random seeds: with identical hyperparameters, most seeds stall near a return of −760 and only a minority swing up and balance. A start-state curriculum does not fix this. Our earlier single-seed hyperparameter sweep could therefore not tell hyperparameter effects apart from seed luck, and we report success rates over seeds instead. This is a working paper; every number is read from the result files in the repository at render time.

In [1]:
import csv, json
from pathlib import Path


import matplotlib.pyplot as plt
from IPython.display import Markdown

R = Path("../results")

def load_csv(p):
    with open(p) as f:
        return list(csv.DictReader(f))

def bench(p):
    p = Path(p)
    return {int(r["envs"]): float(r["steps_per_sec"]) for r in load_csv(p)} if p.exists() else {}

cpu = bench(R / "bench/drone_cpu.csv")
gpu = bench(R / "gpu/bench/drone_gpu.csv")
tpu_summary = json.loads((R / "tpu/summary.json").read_text()) if (R / "tpu/summary.json").exists() else None
tpu = bench(R / "tpu/bench/drone_tpu.csv")
gsum = json.loads((R / "gpu/summary.json").read_text())
ctrl = json.loads((R / "pendulum_controller.json").read_text())
studies = {}
for name in ("small_pop", "curriculum"):
    d = R / "cpu_seeds" / name
    studies[name] = (json.loads((d / "summary.json").read_text()), load_csv(d / "seeds.csv"))
def n_solved(s): return round(s["success_rate"] * s["n_seeds"])
peak_gpu = max(gpu.items(), key=lambda kv: kv[1])
peak_cpu = max(cpu.items(), key=lambda kv: kv[1])
d = gsum["drone"]

## 1 Introduction

Evolution strategies optimise a policy by perturbing its parameters, scoring each perturbed copy on whole episodes and moving the parameters towards the copies that did well ([Wierstra et al. 2014](#ref-wierstra2014); [Salimans et al. 2017](#ref-salimans2017)). ES needs no gradient through the environment, no value function and no replay buffer. Its cost is almost entirely environment simulation, and that simulation is parallel across the population. JAX ([Bradbury et al. 2018](#ref-jax2018)) makes this parallelism cheap to write: if the environment is a pure function, `jax.vmap` turns one episode into thousands without changing the environment code. The same idea underlies JAX environment and ES libraries such as Brax ([Freeman et al. 2021](#ref-freeman2021brax)), gymnax ([Lange 2022](#ref-lange2022gymnax)) and evosax ([Lange 2023](#ref-lange2023evosax)).

This working paper documents a small, self-contained implementation (`swarm_es`) and asks two questions:

1.  **Throughput.** How does environment throughput scale with the number of parallel environments on a CPU, a T4 GPU and a TPU?
2.  **Reliability.** On the pendulum swing-up, does ES solve the task reliably, or does the outcome depend on the random seed?

The second question came from a failure. Our first pendulum runs plateaued near a return of −760, far from the ≈ −150 that counts as solved. A hyperparameter sweep from a single seed did not explain the plateau. We show that the environment is solvable, that the plateau is a seed-dependent local optimum, and that results should therefore be reported over seeds ([Henderson et al. 2018](#ref-henderson2018)).

## 2 Method

### 2.1 Evolution strategies

We use OpenAI-ES ([Salimans et al. 2017](#ref-salimans2017)). The search distribution is an isotropic Gaussian with mean $\theta$ and scale $\sigma$. Each generation samples $n/2$ noise vectors $\epsilon_i\sim\mathcal N(0,I)$ and evaluates the antithetic population $\theta\pm\sigma\epsilon_i$. Fitness $F_i$ is the average return over $k$ episodes. All members share the same $k$ episode seeds (common random numbers), so ranking compares policies rather than luck. Fitness is replaced by centred ranks $u_i\in[-0.5,0.5]$, and the search gradient

<span id="eq-grad">$$
g = \frac{1}{n\sigma}\sum_{i=1}^{n} u_i\,\epsilon_i
 \qquad(1)$$</span>

is followed with Adam ([Kingma and Ba 2015](#ref-kingma2015adam)), with optional weight decay and per-generation decay of the learning rate and of $\sigma$. The policy is a tanh MLP (two hidden layers of 32 units) whose weights are flattened into one vector. After every generation the mean $\theta$ is evaluated on fresh episodes of the full task. This *mean-policy return* is the number we report.

### 2.2 Everything is a JAX function

An environment is a `NamedTuple` state with pure `reset` and `step` functions. One episode is a fixed-length `lax.scan`. Episodes that end early are frozen and their later rewards masked, so every episode has the same shape. A generation is `vmap(vmap(rollout))` over policies and episodes, so with $n=2048$ and $k=5$ one call simulates 10,240 environments. Ten generations are grouped in one `lax.scan` inside one `jit`, so Python runs only between chunks.

### 2.3 Tasks

**Drone.** This is an original planar quadrotor with two rotors. It can only move sideways by tilting, through uneven thrust. Wind gusts follow an Ornstein–Uhlenbeck process ([Uhlenbeck and Ornstein 1930](#ref-uhlenbeck1930)). Reaching a target within 30 cm gives +5 and spawns a new target. Leaving the 8 × 8 m arena or tipping past 90° ends the episode. Episodes last 500 steps (10 s at 50 Hz).

**Pendulum.** The dynamics and cost are those of Gym’s `Pendulum-v1` ([Brockman et al. 2016](#ref-brockman2016gym)), so returns are comparable: a random policy scores about −1200 and a solved one about −150. Episodes last 200 steps and start from a uniformly random angle. An optional curriculum scales the start range by a difficulty $d\in(0,1]$ that grows linearly from 0.1 to 1. Even then, the mean policy is always evaluated at $d=1$.

### 2.4 Experiments

- **Throughput.** Each environment gets its own perturbed parameters, as in a real population. We time one full episode per environment in one jitted call, for 1 to 65,536 environments (262,144 on the TPU).
- **Solvability check.** A hand-written controller with no learning: energy pumping away from upright ([Åström and Furuta 2000](#ref-astrom2000)) and a PD stabiliser near upright, evaluated on 100 episodes.
- **Seed studies.** One config is trained from 8 seeds (CPU) or 12 seeds (TPU). A seed counts as *solved* if its final mean policy beats −200 on 100 fresh episodes. The CPU config is $n=512$, $k=4$, 300 generations, $\sigma=0.05$, learning rate 0.02, weight decay 0.005, with and without a 150-generation curriculum.

## 3 Results

### 3.1 Throughput

In [2]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for label, data, style in [("CPU (Colab)", cpu, "s--"), ("T4 GPU", gpu, "o-")] + ([(f"TPU ({tpu_summary['device']})", tpu, "^-")] if tpu else []):
    xs = sorted(data)
    ax.loglog(xs, [data[x] for x in xs], style, label=label)
ax.set_xlabel("parallel environments"); ax.set_ylabel("environment steps / s")
ax.grid(alpha=0.3, which="both"); ax.legend(); fig.tight_layout(); plt.show()

In [3]:
n_big = max(set(cpu) & set(gpu))
Markdown(f"On the CPU, throughput peaks at {peak_cpu[1]/1e3:,.0f}k steps/s with {peak_cpu[0]:,} environments and then falls as the batch outgrows the cache. "
         f"On the T4 it keeps rising to {peak_gpu[1]/1e6:.1f}M steps/s at {peak_gpu[0]:,} environments, "
         f"{gpu[n_big]/cpu[n_big]:.0f}× the CPU at the same size (@fig-throughput). "
         f"With a single environment the GPU is slower than the CPU ({gpu[1]/1e3:.0f}k vs {cpu[1]/1e3:.0f}k steps/s), because launch overhead dominates. "
         + (f"On the TPU ({tpu_summary['device']}) the peak is {max(tpu.values())/1e6:.0f}M steps/s at {max(tpu, key=tpu.get):,} environments, "
            f"{max(tpu.values())/peak_gpu[1]:.0f}× the T4 peak. Above that size it drops sharply, to {tpu[max(tpu)]/1e6:.0f}M steps/s at {max(tpu):,} environments "
            f"(still {tpu[max(tpu)]/peak_gpu[1]:.1f}× the T4 peak). Each environment carries its own copy of the policy parameters, "
            "so we attribute the cliff to the working set no longer fitting in on-chip memory. This is a hypothesis; we have not profiled it."
            if tpu else "TPU numbers will appear here once `notebooks/tpu_run.ipynb` has been run."))

### 3.2 Drone

In [4]:
Markdown(f"With $n=2048$, $k=5$ (10,240 environments per generation) and 500 generations on the T4, the drone goes from a return of {d['start']:.1f} "
         f"(crashing, no targets) to {d['final']:.1f} with {d['final_hits']:.1f} targets per 10-second episode "
         f"(average of the last 10 generations: {d['last10_avg']:.1f}, {d['last10_hits_avg']:.1f} targets). "
         f"The best generation was {d['best_gen']} ({d['best']:.1f}). The whole run took {gsum['drone_time']/60:.1f} minutes. "
         "Most of the gain comes in the first ~40 generations; after that the return plateaus.")

### 3.3 The pendulum is solvable, but ES is bimodal

In [5]:
Markdown(f"The hand-written controller scores {ctrl['mean']:.1f} ± {ctrl['std']:.1f} (median {ctrl['median']:.1f}) over {ctrl['episodes']} episodes, "
         "well inside the solved range. The environment is therefore not the cause of the −760 plateau.")

In [6]:
rows = ["| Study | Device | Seeds solved | Median final return | Best final return |", "|:--|:--|:--:|--:|--:|"]
for name, (s, _) in studies.items():
    rows.append(f"| {name.replace('_', ' ')} | CPU | {n_solved(s)} / {s['n_seeds']} | {s['final_eval_median']:.1f} | {s['final_eval_best']:.1f} |")
if tpu_summary:
    for name, s in tpu_summary["studies"].items():
        rows.append(f"| {name.replace('_', ' ')} | TPU | {n_solved(s)} / {s['n_seeds']} | {s['final_eval_median']:.1f} | {s['final_eval_best']:.1f} |")
Markdown("\n".join(rows))

In [7]:
s, seed_rows = studies["small_pop"]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for r in seed_rows:
    h = load_csv(R / "cpu_seeds/small_pop/runs" / f"seed{r['seed']}" / "pendulum/history.csv") if (R / "cpu_seeds/small_pop/runs").exists() else None
    if h is None:
        break
    ys = [float(x["mean_policy_return"]) for x in h]
    ok = int(r["success"])
    ax.plot(range(1, len(ys) + 1), ys, color="tab:green" if ok else "tab:gray", lw=1.4 if ok else 0.9, alpha=0.9 if ok else 0.6)
ax.axhline(-200, ls="--", color="tab:red", lw=1)
ax.set_xlabel("generation"); ax.set_ylabel("mean-policy return"); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

In [8]:
s0, r0 = studies["small_pop"]; s1, r1 = studies["curriculum"]
stuck = [float(r["final_eval"]) for r in r0 if float(r["final_eval"]) <= -600]
escaped = [float(r["final_eval"]) for r in r0 if float(r["final_eval"]) > -600]
near = [x for x in escaped if x < -200]
Markdown(f"With identical hyperparameters, {n_solved(s0)} of {s0['n_seeds']} seeds solve the swing-up (@tbl-seeds, @fig-seeds). "
         f"{len(escaped)} of {s0['n_seeds']} leave the plateau at all"
         + (f" ({len(near)} of them narrowly misses the threshold, at {near[0]:.0f})" if len(near) == 1 else "") + ". "
         f"The other {len(stuck)} finish between {min(stuck):.0f} and {max(stuck):.0f}: "
         "a policy that spins or hangs near the bottom instead of balancing. "
         f"The curriculum does not help: {n_solved(s1)} of {s1['n_seeds']} seeds solved, median {s1['final_eval_median']:.0f}. "
         "Starting near upright teaches balancing, but balancing does not lead to swinging up, so the hard part of the task is still unsolved when the start range widens.")

In exploratory runs we also varied $\sigma$ (0.05–0.1, with and without decay), the learning rate, the number of episodes per member (4–16), the network width (32 or 64 units) and weight decay (0 or 0.005). None of these consistently removed the plateau. The same setting solved the task from one seed and failed from another. Even changing how often the training loop logs, which changes how the random-key stream is split, flipped individual outcomes. Our earlier sweep trained every configuration from the same single seed, so its ranking of configurations mostly reflects which random streams happened to escape the plateau.

## 4 Discussion

**Speed is not the bottleneck; exploration is.** On a T4 a pendulum run takes well under a minute, yet most runs end in the same local optimum. For ES on this task, a run’s outcome is a random variable and should be reported as a success rate with its spread, as argued for deep RL by Henderson et al. ([2018](#ref-henderson2018)).

**What would likely help.** The plateau is a known difficulty of swing-up, which needs an energy-pumping phase before balancing ([Åström and Furuta 2000](#ref-astrom2000)). Options we have not yet tested:

- *Parallel restarts.* Run many independent ES instances, which is itself a `vmap`, and keep the best. At a success rate of $p$ per seed, $m$ restarts succeed with probability $1-(1-p)^m$.
- *Larger or adaptive exploration*, for example novelty bonuses or a $\sigma$ schedule driven by stagnation.
- *Better curricula*, such as reverse curricula that start from states the current policy can already reach ([Florensa et al. 2017](#ref-florensa2017reverse)) instead of a fixed angle schedule ([Bengio et al. 2009](#ref-bengio2009curriculum)).
- *Simpler policies.* Linear policies with random search are competitive on many control tasks ([Mania et al. 2018](#ref-mania2018)).

**Limitations.** Two small tasks; one policy architecture; at most 12 seeds per setting, so the success rates are coarse; the drone was trained from a single seed. The TPU throughput cliff is not profiled.

## 5 Reproducibility

All code is at <https://github.com/shiroonigami23-ui/jax-swarm-es>. This paper is a Quarto document whose numbers, tables and figures are computed from the files in `results/` each time it is rendered. The commands are:

- `python -m swarm_es.bench`: throughput.
- `python -m swarm_es.baselines`: the hand-written controller.
- `python -m swarm_es.seeds`: seed studies.
- `notebooks/tpu_run.ipynb`, which calls `colab_run.py --tpu`: the full TPU study.

## References

Åström, Karl Johan, and Katsuhisa Furuta. 2000. “Swinging up a Pendulum by Energy Control.” *Automatica* 36 (2): 287–95.

Bengio, Yoshua, Jérôme Louradour, Ronan Collobert, and Jason Weston. 2009. “Curriculum Learning.” *Proceedings of the 26th International Conference on Machine Learning*, 41–48.

Bradbury, James, Roy Frostig, Peter Hawkins, et al. 2018. *JAX: Composable Transformations of Python+NumPy Programs*. Released. <http://github.com/jax-ml/jax>.

Brockman, Greg, Vicki Cheung, Ludwig Pettersson, et al. 2016. “OpenAI Gym.” *arXiv Preprint arXiv:1606.01540*.

Florensa, Carlos, David Held, Markus Wulfmeier, Michael Zhang, and Pieter Abbeel. 2017. “Reverse Curriculum Generation for Reinforcement Learning.” *Proceedings of the 1st Conference on Robot Learning*.

Freeman, C. Daniel, Erik Frey, Anton Raichuk, Sertan Girgin, Igor Mordatch, and Olivier Bachem. 2021. “Brax – a Differentiable Physics Engine for Large Scale Rigid Body Simulation.” *arXiv Preprint arXiv:2106.13281*.

Henderson, Peter, Riashat Islam, Philip Bachman, Joelle Pineau, Doina Precup, and David Meger. 2018. “Deep Reinforcement Learning That Matters.” *Proceedings of the AAAI Conference on Artificial Intelligence*.

Kingma, Diederik P., and Jimmy Ba. 2015. “Adam: A Method for Stochastic Optimization.” *International Conference on Learning Representations*.

Lange, Robert Tjarko. 2022. *Gymnax: A JAX-Based Reinforcement Learning Environment Library*. Released. <http://github.com/RobertTLange/gymnax>.

Lange, Robert Tjarko. 2023. “Evosax: JAX-Based Evolution Strategies.” *Proceedings of the Companion Conference on Genetic and Evolutionary Computation*.

Mania, Horia, Aurelia Guy, and Benjamin Recht. 2018. “Simple Random Search of Static Linear Policies Is Competitive for Reinforcement Learning.” *Advances in Neural Information Processing Systems*.

Salimans, Tim, Jonathan Ho, Xi Chen, Szymon Sidor, and Ilya Sutskever. 2017. “Evolution Strategies as a Scalable Alternative to Reinforcement Learning.” *arXiv Preprint arXiv:1703.03864*. <https://arxiv.org/abs/1703.03864>.

Uhlenbeck, George E., and Leonard S. Ornstein. 1930. “On the Theory of the Brownian Motion.” *Physical Review* 36 (5): 823–41.

Wierstra, Daan, Tom Schaul, Tobias Glasmachers, Yi Sun, Jan Peters, and Jürgen Schmidhuber. 2014. “Natural Evolution Strategies.” *Journal of Machine Learning Research* 15: 949–80.